# Holidays Events Exploratory Data Analysis (EDA)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Find the project root containing pyproject.toml
root = Path.cwd()
while root.name and not (root / 'pyproject.toml').exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.append(str(root))

from configs.config import RAW_DIR

## 1. Basic Structure
Shape, columns, and unique values of `type`, `locale`, `transferred`.

In [2]:
holidays = pd.read_csv(RAW_DIR / 'holidays_events.csv')
holidays['date'] = pd.to_datetime(holidays['date'])

print(f"Shape: {holidays.shape}")
print(f"Columns: {list(holidays.columns)}")
print(f"Dtypes:\n{holidays.dtypes}")
print(f"\nMissing values:\n{holidays.isnull().sum()}")

print(f"\nUnique values - type:\n{holidays['type'].value_counts()}")
print(f"\nUnique values - locale:\n{holidays['locale'].value_counts()}")
print(f"\nUnique values - transferred:\n{holidays['transferred'].value_counts()}")

Shape: (350, 6)
Columns: ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']
Dtypes:
date           datetime64[ns]
type                   object
locale                 object
locale_name            object
description            object
transferred              bool
dtype: object

Missing values:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

Unique values - type:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

Unique values - locale:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

Unique values - transferred:
transferred
False    338
True      12
Name: count, dtype: int64


## 2. transferred=True vs type='Transfer' — matched-pairs hypothesis check


In [3]:
transferred_true = holidays[holidays['transferred'] == True]
transfer_rows = holidays[holidays['type'] == 'Transfer']

print(f"Number of transferred=True rows: {len(transferred_true)}")
print(f"Of which type='Holiday': {(transferred_true['type'] == 'Holiday').sum()} / {len(transferred_true)}")
print(f"Number of type='Transfer' rows: {len(transfer_rows)}")

Number of transferred=True rows: 12
Of which type='Holiday': 12 / 12
Number of type='Transfer' rows: 12


## 3. Classification per locked logic


In [4]:
mask_working_day_transferred_holiday = (holidays['type'] == 'Holiday') & (holidays['transferred'] == True)
mask_actual_holiday_not_transferred = (holidays['type'] == 'Holiday') & (holidays['transferred'] == False)
mask_actual_holiday_transfer = holidays['type'] == 'Transfer'

print(f"type='Holiday' & transferred=True (normal working day — original holiday moved): "
      f"{mask_working_day_transferred_holiday.sum()}")
print(f"type='Holiday' & transferred=False (regular holiday, not moved — still an actual day off): "
      f"{mask_actual_holiday_not_transferred.sum()}")
print(f"type='Transfer' (actual observed day off, holiday moved to this date): "
      f"{mask_actual_holiday_transfer.sum()}")

type='Holiday' & transferred=True (normal working day — original holiday moved): 12
type='Holiday' & transferred=False (regular holiday, not moved — still an actual day off): 209
type='Transfer' (actual observed day off, holiday moved to this date): 12


## 4. Does locale_name match stores.city/state?


In [5]:
stores = pd.read_csv(RAW_DIR / 'stores.csv')

locale_names = set(holidays['locale_name'].unique())
cities = set(stores['city'].unique())
states = set(stores['state'].unique())

unmatched_locale = locale_names - cities - states - {'Ecuador'}
cities_without_local_holiday = cities - locale_names

print(f"Number of unique locale_name values: {len(locale_names)}")
print(f"locale_name not matching any city/state/'Ecuador' (expected empty): {unmatched_locale}")
print(f"Cities in stores.csv with no Local holiday: {sorted(cities_without_local_holiday)}")

Number of unique locale_name values: 24
locale_name not matching any city/state/'Ecuador' (expected empty): set()
Cities in stores.csv with no Local holiday: ['Babahoyo', 'Daule', 'Playas']


## 5. Overlapping holidays on the same date (31 days breakdown)
Analyze the 31 dates that have multiple holiday records to see:
1. How many dates are "different locales" (i.e. disjoint geography, resolved automatically when merged by store).
2. How many dates are "same store overlap" (i.e. at least one store gets multiple holidays on this date, requiring logic to merge).

In [6]:
# Find dates with multiple holiday records
date_counts = holidays.groupby('date').size()
dup_dates = date_counts[date_counts > 1].index

print(f"Total dates with multiple holiday records: {len(dup_dates)}\n")

# Check overlap for each store
overlap_results = []
for date in dup_dates:
    date_events = holidays[holidays['date'] == date]
    
    # Store level mapping
    store_event_counts = []
    for _, store in stores.iterrows():
        store_nbr = store['store_nbr']
        city = store['city']
        state = store['state']
        
        # Check which events apply to this store
        app_events = []
        for _, event in date_events.iterrows():
            loc = event['locale']
            loc_name = event['locale_name']
            
            if loc == 'National':
                app_events.append(event)
            elif loc == 'Regional' and loc_name == state:
                app_events.append(event)
            elif loc == 'Local' and loc_name == city:
                app_events.append(event)
        
        store_event_counts.append((store_nbr, len(app_events), app_events))
    
    # Analyze store counts
    max_events = max(cnt for _, cnt, _ in store_event_counts)
    same_store_dup = any(cnt > 1 for _, cnt, _ in store_event_counts)
    
    overlap_results.append({
        'date': date,
        'raw_events_count': len(date_events),
        'max_events_per_store': max_events,
        'same_store_dup': same_store_dup,
        'events': [f"{e['locale']}:{e['locale_name']}:{e['description']}" for _, e in date_events.iterrows()]
    })

df_overlap = pd.DataFrame(overlap_results)

# Print Summary
resolved = df_overlap[~df_overlap['same_store_dup']]
unresolved = df_overlap[df_overlap['same_store_dup']]

print(f"Number of dates that automatically resolve (khác locale, max event per store <= 1): {len(resolved)}")
print(f"Number of dates that STILL have duplicates for at least one store (cùng áp dụng cho cùng 1 store): {len(unresolved)}")

print("\n=== RESOLVED DATES (disjoint locales) ===")
for _, row in resolved.iterrows():
    print(f"\nDate: {row['date'].strftime('%Y-%m-%d')} | Events: {row['raw_events_count']}")
    for ev in row['events']:
        print(f"  - {ev}")

print("\n=== UNRESOLVED DATES (same store overlaps) ===")
for _, row in unresolved.iterrows():
    print(f"\nDate: {row['date'].strftime('%Y-%m-%d')} | Events: {row['raw_events_count']} | Max events per store: {row['max_events_per_store']}")
    for ev in row['events']:
        print(f"  - {ev}")

Total dates with multiple holiday records: 31



Number of dates that automatically resolve (khác locale, max event per store <= 1): 12
Number of dates that STILL have duplicates for at least one store (cùng áp dụng cho cùng 1 store): 19

=== RESOLVED DATES (disjoint locales) ===

Date: 2012-06-25 | Events: 3
  - Regional:Imbabura:Provincializacion de Imbabura
  - Local:Latacunga:Cantonizacion de Latacunga
  - Local:Machala:Fundacion de Machala

Date: 2012-07-03 | Events: 2
  - Local:Santo Domingo:Fundacion de Santo Domingo
  - Local:El Carmen:Cantonizacion de El Carmen

Date: 2013-06-25 | Events: 3
  - Regional:Imbabura:Provincializacion de Imbabura
  - Local:Machala:Fundacion de Machala
  - Local:Latacunga:Cantonizacion de Latacunga

Date: 2013-07-03 | Events: 2
  - Local:El Carmen:Cantonizacion de El Carmen
  - Local:Santo Domingo:Fundacion de Santo Domingo

Date: 2014-07-03 | Events: 2
  - Local:El Carmen:Cantonizacion de El Carmen
  - Local:Santo Domingo:Fundacion de Santo Domingo

Date: 2015-06-25 | Events: 3
  - Local:Machala: